# Single Task Submission

This notebook demonstrates the basics of running Prefect tasks on SLURM using `prefect-submitit`.

You'll learn how to:
- Submit a single task to SLURM (or run locally for development)
- Chain task dependencies
- Work with complex return values
- Inspect SLURM job IDs and logs

## Setup

Make sure the Prefect server is running before executing this notebook:
```bash
pixi run prefect-start
```

In [ ]:
from prefect import flow, task
from prefect_submitit import SlurmTaskRunner

In [ ]:
# "local" or "slurm"
MODE = "slurm" 

## Define Tasks

Tasks must be defined at the **module/cell level** (not inside functions) so they can be serialized correctly.

In [ ]:
@task
def add(a: int, b: int) -> int:
    return a + b


@task
def multiply(a: int, b: int) -> int:
    return a * b

## Submit a Single Task

Use `.submit()` to send work to SLURM. The returned future lets you retrieve the result.

In [ ]:
runner = SlurmTaskRunner(execution_mode=MODE, slurm_name="single_task_flow")


@flow(task_runner=runner)
def single_task_flow():
    future = add.submit(1, 2)
    result = future.result()
    print(f"1 + 2 = {result}")
    return result


result = single_task_flow()
assert result == 3

## Task Dependencies

Chain tasks by passing the result of one task as input to another.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="dependency_flow"))
def dependency_flow():
    # Step 1: add 1 + 2
    sum_future = add.submit(1, 2)
    sum_result = sum_future.result()  # == 3

    # Step 2: multiply the sum by 10
    product_future = multiply.submit(sum_result, 10)
    product_result = product_future.result()  # == 30

    print(f"(1 + 2) * 10 = {product_result}")
    return product_result


result = dependency_flow()
assert result == 30

## Complex Return Values

Tasks can return dicts, nested structures, and other serializable objects.

In [ ]:
@task
def return_complex():
    return {
        "status": "ok",
        "values": [1, 2, 3],
        "nested": {"a": True, "b": None},
    }


@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="complex_return_flow"))
def complex_return_flow():
    future = return_complex.submit()
    result = future.result()
    print(f"Got complex result: {result}")
    return result


result = complex_return_flow()
assert result["status"] == "ok"
assert result["values"] == [1, 2, 3]
assert result["nested"]["b"] is None

## Inspect Job ID and Logs

Each future exposes the SLURM job ID and captured stdout/stderr logs.

In [ ]:
@task
def greet(name: str) -> str:
    print(f"Hello, {name}!")  # captured in stdout logs
    return f"greeting for {name}"


@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="inspect_flow"))
def inspect_flow():
    future = greet.submit(name="World")
    result = future.result()

    # The SLURM job ID (or local executor job ID in local mode)
    print(f"Job ID: {future.slurm_job_id}")

    # Captured logs from the worker
    stdout, stderr = future.logs()
    print(f"stdout: {stdout!r}")
    print(f"stderr: {stderr!r}")

    return result


inspect_flow()